# Suspicious Login Detection Using Machine Learning

**IBM SkillsBuild Cyber Security 2026 — Project Submission**

This educational prototype analyzes synthetic authentication events and flags potentially suspicious logins using transparent rules and an Isolation Forest anomaly detector. It does not connect to live systems or use real personal data.

## 1. Objective
Detect unusual login activity from event features such as failed attempts, login hour, unfamiliar IP, new device, and geographic change. The dataset is synthetic so results demonstrate the workflow rather than real-world security performance.

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('Python:', sys.version.split()[0])

## 2. Generate a reproducible synthetic dataset
Normal events are more common; a smaller group is intentionally generated with suspicious characteristics. `label` is used only for illustrative evaluation, not as an input feature.

In [ ]:
n_normal, n_suspicious = 900, 100
def make_events(n, suspicious=False):
    if suspicious:
        return pd.DataFrame({
            'failed_attempts': np.random.randint(3, 10, n),
            'login_hour': np.random.choice([0,1,2,3,4,23], n),
            'new_device': np.random.binomial(1, .8, n),
            'unfamiliar_ip': np.random.binomial(1, .85, n),
            'geo_distance_km': np.random.gamma(5, 180, n),
            'account_age_days': np.random.randint(1, 120, n),
            'label': 1})
    return pd.DataFrame({
        'failed_attempts': np.random.poisson(.35, n).clip(0, 3),
        'login_hour': np.random.choice(np.arange(7, 23), n),
        'new_device': np.random.binomial(1, .12, n),
        'unfamiliar_ip': np.random.binomial(1, .10, n),
        'geo_distance_km': np.random.gamma(1.2, 12, n),
        'account_age_days': np.random.randint(30, 2500, n),
        'label': 0})
df = pd.concat([make_events(n_normal), make_events(n_suspicious, True)], ignore_index=True)
df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
df.head()

In [ ]:
print('Dataset shape:', df.shape)
print('\nClass distribution (0=normal, 1=synthetic suspicious):')
print(df['label'].value_counts().sort_index())
display(df.describe().round(2))

## 3. Exploratory analysis

In [ ]:
df.groupby('label')[['failed_attempts','geo_distance_km','new_device','unfamiliar_ip']].mean().round(2)

In [ ]:
df['label'].value_counts().sort_index().plot(kind='bar', rot=0)
plt.title('Synthetic Login Events by Class')
plt.xlabel('Label (0=normal, 1=suspicious)'); plt.ylabel('Event count')
plt.tight_layout(); plt.show()

## 4. Train and evaluate an anomaly detector
Isolation Forest is trained on a training split without using the `label` feature. Because this is a demonstration dataset with deliberately separated distributions, its results should not be interpreted as expected production accuracy.

In [ ]:
features = ['failed_attempts','login_hour','new_device','unfamiliar_ip','geo_distance_km','account_age_days']
X_train, X_test, y_train, y_test = train_test_split(df[features], df['label'], test_size=.25, random_state=RANDOM_STATE, stratify=df['label'])
model = IsolationForest(n_estimators=200, contamination=0.10, random_state=RANDOM_STATE)
model.fit(X_train)
# IsolationForest predicts -1 for anomalies and +1 for inliers
pred = (model.predict(X_test) == -1).astype(int)
print(classification_report(y_test, pred, target_names=['Normal','Synthetic suspicious'], zero_division=0))
print('Confusion matrix [rows=true, columns=predicted]:\n', confusion_matrix(y_test, pred))

## 5. Transparent rule-based triage
The rules are illustrative and should be calibrated against an organization's baseline. A flag means 'review this event', not 'confirmed attack'.

In [ ]:
def rule_score(event):
    score = 0
    reasons = []
    if event['failed_attempts'] >= 4: score += 2; reasons.append('multiple failed attempts')
    if event['login_hour'] <= 5 or event['login_hour'] >= 23: score += 1; reasons.append('unusual login hour')
    if event['new_device'] == 1: score += 1; reasons.append('new device')
    if event['unfamiliar_ip'] == 1: score += 2; reasons.append('unfamiliar IP')
    if event['geo_distance_km'] >= 300: score += 2; reasons.append('large geographic change')
    return pd.Series({'risk_score': score, 'review_flag': score >= 3, 'reasons': ', '.join(reasons) if reasons else 'no rule triggered'})
demo = pd.DataFrame([
 {'failed_attempts':0,'login_hour':14,'new_device':0,'unfamiliar_ip':0,'geo_distance_km':8,'account_age_days':500},
 {'failed_attempts':6,'login_hour':2,'new_device':1,'unfamiliar_ip':1,'geo_distance_km':850,'account_age_days':20}
])
results = pd.concat([demo, demo.apply(rule_score, axis=1)], axis=1)
results

## 6. Save outputs
The CSV is generated locally and contains only synthetic records.

In [ ]:
df.to_csv('synthetic_login_events.csv', index=False)
results.to_csv('example_triage_results.csv', index=False)
print('Saved synthetic_login_events.csv and example_triage_results.csv')

## 7. Findings, limitations, and next steps
- The pipeline demonstrates feature engineering, anomaly detection, evaluation, and explainable rule-based triage.
- Synthetic distributions are intentionally simplified; real attack data is noisier and may overlap with normal behavior.
- Isolation Forest can produce false positives and false negatives. Thresholds require validation.
- Before any operational use, test on authorized, representative data; protect logs; document retention; review bias and privacy; and require a human analyst to confirm alerts.
- This notebook is an educational prototype, not a production security monitoring system.